# Exercice 4 — Challenger un score de rachat : cadrage, fuite de données, split temporel et dérive

> **VERSION CORRIGÉE** 

##  Contexte métier
La direction Épargne veut un **modèle de probabilité de rachat à 12 mois** pour fiabiliser la projection de liquidité. Un premier prototype affiche une AUC de 0,99 : « bravo ! » dit-on… Votre rôle d'actuaire est de **challenger** ce chiffre : le problème est-il bien posé ? les variables étaient-elles connues à la date de décision ? le protocole de test est-il honnête ? le modèle vieillira-t-il bien en production ? C'est tout le **cycle de vie d'un modèle**, de la formulation au monitoring.

## Objectifs
- Évaluer la qualité d'une **formulation de problème** avec les quatre questions du cadrage
- Repérer et exclure les variables qui **fuient le futur** (data leakage), puis mesurer l'effet sur la performance
- Comparer un **split aléatoire** et un **split temporel** : lequel est honnête sur des données chronologiques ?
- Mettre en place un **indicateur de dérive (PSI)** pour le monitoring

##  Rappels théoriques mobilisés (slides de la session)
- *Formuler le problème : quatre questions avant toute donnée* — décision, cible, métrique, usage
- *Le cycle de vie d'un modèle* — formulation, données, entraînement, validation, déploiement, monitoring
- *Concepts clés* — features disponibles à la date de décision, label sur horizon explicite, généralisation
- *Risques transverses* — fuite de données, biais d'échantillon, sur-évaluation, dérive
- *Risque empirique, risque réel* — un split aléatoire sur données temporelles surestime la performance

##  Mini-dictionnaire & prérequis
| Élément | Signification / usage |
|---|---|
| Feature | variable **connue à la date de décision** (31/12/N) |
| Label / cible | `rachat_12m` = 1 si rachat total dans les 12 mois suivant le 31/12/N |
| Fuite de données | une variable informée par le futur s'invite dans les features → performance mirifique, inutilisable en production |
| Split temporel | train sur les années passées, test sur l'année suivante (comme en production) |
| Calibration globale | écart entre probabilité moyenne prédite et taux observé (négatif = sous-estimation) |
| PSI | Population Stability Index = Σ (a_i − e_i)·ln(a_i / e_i) ; e = proportions de référence, a = proportions actuelles, par classes |
| Seuils PSI usuels | < 0,10 stable · 0,10–0,25 à surveiller · > 0,25 dérive significative |
| `evaluer(train, test, features)` | fonction fournie : entraîne une régression logistique et renvoie AUC, Brier, p_moy, taux_obs |

In [ ]:
# Outil d'auto-vérification — NE PAS MODIFIER
import numpy as np, pandas as pd, matplotlib.pyplot as plt, hashlib
_resultats = {}
def verifier(nom, condition, indice=""):
    """Évalue `condition` (une fonction sans argument) ; affiche ✅ ou ❌ + indice."""
    try: ok = bool(condition())
    except Exception: ok = False
    _resultats[nom] = ok
    print(("✅ " if ok else "❌ ") + nom + ("" if ok or not indice else "\n    💡 " + indice))
def bilan():
    n, k = len(_resultats), sum(_resultats.values())
    print(f"\n🏁 Score : {k}/{n} vérifications réussies")
    print("   Bravo, exercice terminé !" if k == n and n > 0 else "   Reprenez les ❌ ci-dessus (relancez la cellule concernée puis `bilan()`).")
def _h(s): return hashlib.sha1(s.encode()).hexdigest()[:8]
%matplotlib inline

---
## Partie A — Bien poser le problème (avant toute donnée)

Quatre formulations circulent dans l'entreprise. Une formulation bien posée répond à **quatre questions** : 
- `decision` : quelle décision métier le modèle éclaire-t-il ?
- `cible` : variable, horizon et population précis ?
- `metrique` : quelle métrique de **succès métier** (pas seulement technique) ?
- `usage` : aide à la décision, automatisation partielle ou exploration ?

> **A.** « Faire du machine learning sur les rachats. »
>
> **B.** « Estimer la probabilité de rachat total à 12 mois de chaque contrat en cours au 31/12. »
>
> **C.** « Estimer au 31/12 la probabilité de rachat total à 12 mois de chaque contrat en cours, afin de fiabiliser la projection de liquidité ; succès = erreur de flux projeté réduite de 20 % vs la méthode actuelle. »
>
> **D.** « C, utilisé en aide à la décision avec validation humaine avant toute action commerciale. »

**Q1.** Pour chaque formulation, listez dans `manques` les éléments **absents** parmi `"decision"`, `"cible"`, `"metrique"`, `"usage"` (liste vide `[]` si rien ne manque).

In [2]:
manques = {
    "A": ["decision", "cible", "metrique", "usage"],
    "B": ["decision", "metrique", "usage"],
    "C": ["usage"],
    "D": [],
}

In [3]:
_att = {"A": 'a9ed627e',
        "B": 'eb21890c',
        "C": 'f12daa55',
        "D": '97d170e1'}
for _k in "ABCD":
    verifier(f"Q1 · formulation {_k}", lambda: _h(str(sorted(manques[_k]))) == _att[_k], "relisez les 4 questions : décision, cible, métrique, usage")

✅ Q1 · formulation A
✅ Q1 · formulation B
✅ Q1 · formulation C
✅ Q1 · formulation D


**Correction** — A est le contre-exemple (rien n'est défini). B fixe la cible (variable, horizon, population) mais ne dit ni pourquoi (décision), ni comment juger le succès (métrique), ni le niveau d'autonomie (usage). C ajoute décision et métrique ; seul D précise l'usage — le niveau de risque en dépend. **La première décision n'est pas l'algorithme.**

## Partie B — La fuite de données : pourquoi l'AUC de 0,99 est suspecte

Voici l'historique 2018-2023 (1 500 contrats par année) et le **dictionnaire de données**. Exécutez la cellule (ne rien modifier).

In [4]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, brier_score_loss
from sklearn.model_selection import train_test_split

# ---------- Génération des données (ne pas modifier) ----------
rng = np.random.default_rng(7)
decalage = {2018: 0.0, 2019: 0.0, 2020: -0.1, 2021: 0.0, 2022: 0.9, 2023: 1.9}
blocs = []
for annee in range(2018, 2024):
    n = 1500
    age = np.clip(rng.normal(52 + 0.4 * (annee - 2018), 12, n), 25, 88).round()
    anciennete = np.clip(rng.gamma(2.5, 3.0, n), 0.5, 30).round(1)
    encours = np.exp(rng.normal(10.3 + 0.01 * (age - 52), 0.6, n)).round(-2)
    ecart_taux = rng.normal(decalage[annee], 0.6, n).round(2)
    canal = rng.choice(["agent", "courtage", "bancassurance"], n, p=[0.5, 0.2, 0.3])
    logit = (-3.0 + 0.5 * ecart_taux - 0.05 * anciennete + 0.012 * (age - 52)
             + 0.3 * (canal == "courtage") + 0.6 * (annee == 2023))
    rachat = rng.binomial(1, 1 / (1 + np.exp(-logit)))
    blocs.append(pd.DataFrame({
        "annee": annee, "age": age, "anciennete": anciennete, "encours": encours, "ecart_taux": ecart_taux,
        "canal_courtage": (canal == "courtage").astype(int), "canal_banque": (canal == "bancassurance").astype(int),
        "encours_fin_horizon": np.where(rachat == 1, 0.0, (encours * rng.normal(1.02, 0.03, n)).round(-2)),
        "nb_appels_horizon": rng.poisson(0.3 + 2.5 * rachat),
        "rachat_12m": rachat}))
df = pd.concat(blocs, ignore_index=True)

dico = pd.DataFrame([
    ("annee",              "période", "avant décision", "Année N de la date de décision (31/12/N) — identifiant de période"),
    ("age",                "feature", "avant décision", "Âge de l'assuré au 31/12/N"),
    ("anciennete",         "feature", "avant décision", "Ancienneté du contrat (années) au 31/12/N"),
    ("encours",            "feature", "avant décision", "Encours (€) au 31/12/N"),
    ("ecart_taux",         "feature", "avant décision", "Taux de marché − taux servi (points) au 31/12/N"),
    ("canal_courtage",     "feature", "avant décision", "1 si distribué par courtage"),
    ("canal_banque",       "feature", "avant décision", "1 si distribué par bancassurance"),
    ("encours_fin_horizon","feature", "après décision", "Encours (€) au 31/12/N+1"),
    ("nb_appels_horizon",  "feature", "après décision", "Appels au service client pendant l'année N+1"),
    ("rachat_12m",         "cible",   "après décision", "1 si rachat total entre le 01/01 et le 31/12/N+1"),
], columns=["variable", "role", "moment", "description"])

def evaluer(train, test, features):
    """Entraîne une régression logistique (standardisation DANS le pipeline, donc apprise sur le train)."""
    mod = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    mod.fit(train[features], train["rachat_12m"])
    p = mod.predict_proba(test[features])[:, 1]
    return {"auc": roc_auc_score(test["rachat_12m"], p), "brier": brier_score_loss(test["rachat_12m"], p),
            "p_moy": p.mean(), "taux_obs": test["rachat_12m"].mean()}

features_toutes = dico[dico.role == "feature"].variable.tolist()   # ⚠ candidats, pas forcément tous légitimes
print(df.shape, "| taux de rachat moyen par année :", df.groupby("annee").rachat_12m.mean().round(3).to_dict())
dico

(9000, 10) | taux de rachat moyen par année : {2018: 0.031, 2019: 0.035, 2020: 0.037, 2021: 0.043, 2022: 0.051, 2023: 0.16}


,variable,role,moment,description
0,annee,période,avant décision,Année N de la date de décision (31/12/N) — ide...
1,age,feature,avant décision,Âge de l'assuré au 31/12/N
2,anciennete,feature,avant décision,Ancienneté du contrat (années) au 31/12/N
3,encours,feature,avant décision,Encours (€) au 31/12/N
4,ecart_taux,feature,avant décision,Taux de marché − taux servi (points) au 31/12/N
5,canal_courtage,feature,avant décision,1 si distribué par courtage
6,canal_banque,feature,avant décision,1 si distribué par bancassurance
7,encours_fin_horizon,feature,après décision,Encours (€) au 31/12/N+1
8,nb_appels_horizon,feature,après décision,Appels au service client pendant l'année N+1
9,rachat_12m,cible,après décision,1 si rachat total entre le 01/01 et le 31/12/N+1


**Q2.** (a) Construisez `features_ok` : les features **disponibles avant la date de décision** (utilisez le dictionnaire). (b) Avec un split **aléatoire** (`tr`, `te` fournis), évaluez `res_fuite` (toutes les features candidates) et `res_sain` (`features_ok`).

<details><summary>💡 Indice 1</summary>

Filtrez le DataFrame `dico` sur `role == 'feature'` **et** `moment == 'avant décision'`, puis `.variable.tolist()`.

</details>

<details><summary>💡 Indice 2</summary>

`res_fuite = evaluer(tr, te, features_toutes)` ; `res_sain = evaluer(tr, te, features_ok)`.

</details>

In [5]:
tr, te = train_test_split(df, test_size=0.25, random_state=0)
features_ok = dico[(dico.role == "feature") & (dico.moment == "avant décision")].variable.tolist()
res_fuite = evaluer(tr, te, features_toutes)
res_sain = evaluer(tr, te, features_ok)
print(f"AUC avec fuite = {res_fuite['auc']:.3f} | AUC sans fuite = {res_sain['auc']:.3f}")

AUC avec fuite = 1.000 | AUC sans fuite = 0.680


In [6]:
verifier("Q2a · 6 features légitimes", lambda: sorted(features_ok) == sorted(["age","anciennete","encours","ecart_taux","canal_courtage","canal_banque"]), "excluez les variables « après décision »")
verifier("Q2b · l'AUC avec fuite est irréaliste (> 0,95)", lambda: res_fuite["auc"] > 0.95)
verifier("Q2b · l'AUC honnête est modeste (0,55–0,85)", lambda: 0.55 < res_sain["auc"] < 0.85)

✅ Q2a · 6 features légitimes
✅ Q2b · l'AUC avec fuite est irréaliste (> 0,95)
✅ Q2b · l'AUC honnête est modeste (0,55–0,85)


**Q2c · QCM.** Pourquoi `encours_fin_horizon` est-elle une fuite ?
- **A.** Elle est trop corrélée aux autres variables
- **B.** Elle vaut 0 quand le contrat est racheté : elle contient la réponse et n'existe pas au 31/12/N
- **C.** Elle a des valeurs manquantes

Puis : `nb_appels_horizon` est-elle aussi une fuite ? (`"oui"` / `"non"`).

In [7]:
reponse_q2c = "B"
appels_est_fuite = "oui"

In [8]:
verifier("Q2c · raison de la fuite", lambda: reponse_q2c == "B")
verifier("Q2c · nb_appels_horizon est aussi une fuite", lambda: appels_est_fuite == "oui", "elle est observée pendant N+1, donc inconnue au 31/12/N")

✅ Q2c · raison de la fuite
✅ Q2c · nb_appels_horizon est aussi une fuite


**Correction** — L'AUC passe de ≈ 1,00 (avec fuite) à ≈ 0,68 (honnête). La seconde fuite est **subtile** : les appels sont plus fréquents quand le client s'apprête à racheter, mais ne sont connus qu'après. *Test de réflexe : « cette information existerait-elle le jour où je dois prendre la décision ? »*

## Partie C — Split aléatoire ou temporel ?

En production, le modèle entraîné jusqu'en 2022 sera appliqué à **2023**. Comparons deux protocoles d'évaluation (avec `features_ok`) :
- **aléatoire** : `tr`/`te` de la partie B ;
- **temporel** : train = années ≤ 2022, test = année 2023.

**Q3.** Calculez `res_alea` et `res_temp` avec `evaluer`, puis `ecart_alea` et `ecart_temp` = `p_moy − taux_obs` (négatif ⇒ sous-estimation). Quel protocole est honnête ? (`split_honnete` = `"aleatoire"` ou `"temporel"`).

<details><summary>💡 Indice 1</summary>

`train_t = df[df.annee <= 2022]` ; `test_t = df[df.annee == 2023]`.

</details>

In [9]:
train_t, test_t = df[df.annee <= 2022], df[df.annee == 2023]
res_alea = evaluer(tr, te, features_ok)
res_temp = evaluer(train_t, test_t, features_ok)
ecart_alea = res_alea["p_moy"] - res_alea["taux_obs"]
ecart_temp = res_temp["p_moy"] - res_temp["taux_obs"]
split_honnete = "temporel"
print(f"Aléatoire : prédit {res_alea['p_moy']:.1%} vs observé {res_alea['taux_obs']:.1%}")
print(f"Temporel  : prédit {res_temp['p_moy']:.1%} vs observé {res_temp['taux_obs']:.1%}")

Aléatoire : prédit 6.1% vs observé 5.7%
Temporel  : prédit 8.9% vs observé 16.0%


In [10]:
verifier("Q3 · split temporel : bonnes tailles", lambda: len(train_t) == 7500 and len(test_t) == 1500)
verifier("Q3 · écart du split aléatoire", lambda: np.isclose(ecart_alea, 0.00447, atol=2e-3))
verifier("Q3 · écart du split temporel", lambda: np.isclose(ecart_temp, -0.070938, atol=2e-3))
verifier("Q3 · le split temporel révèle la sous-estimation", lambda: ecart_temp < -0.04 and abs(ecart_temp) > 5 * abs(ecart_alea))
verifier("Q3 · protocole honnête", lambda: split_honnete == "temporel", "le modèle sera utilisé sur le futur")

✅ Q3 · split temporel : bonnes tailles
✅ Q3 · écart du split aléatoire
✅ Q3 · écart du split temporel
✅ Q3 · le split temporel révèle la sous-estimation
✅ Q3 · protocole honnête


**Correction & lecture actuarielle** — Avec le split aléatoire, le modèle voit déjà des contrats 2023 pendant l'entraînement : il paraît bien calibré (écart de ≈ +0,4 pt). Avec le split temporel, il prédit ≈ 9 % de rachats là où 16 % sont observés : une **sous-estimation de 7 points** — sur un portefeuille, c'est un trou de liquidité. *Le protocole de séparation compte autant que l'algorithme.*

## Partie D — Surveiller la dérive en production (PSI)

Le PSI compare la distribution d'une variable en production à celle de référence (le train). **Q4.** Complétez `psi(ref, new, bins=10)` : les bornes sont les déciles de `ref` ; `e` = proportions de référence, `a` = proportions de `new`. Calculez `psi_vars` pour `age`, `anciennete`, `encours`, `ecart_taux` (référence = `train_t`, production = `test_t`), puis `statut` (`"stable"`, `"à surveiller"`, `"dérive"`) selon les seuils du dictionnaire.

<details><summary>💡 Indice 1</summary>

`a = np.histogram(new, bins=bornes)[0] / len(new)`

</details>

<details><summary>💡 Indice 2</summary>

Retour : `np.sum((a - e) * np.log(a / e))`

</details>

<details><summary>💡 Indice 3</summary>

Statut : `< 0.10` → stable ; `< 0.25` → à surveiller ; sinon dérive.

</details>

In [11]:
def psi(ref, new, bins=10):
    bornes = np.quantile(ref, np.linspace(0, 1, bins + 1))
    bornes[0], bornes[-1] = -np.inf, np.inf
    e = np.histogram(ref, bins=bornes)[0] / len(ref)
    a = np.histogram(new, bins=bornes)[0] / len(new)
    e, a = np.clip(e, 1e-6, None), np.clip(a, 1e-6, None)
    return np.sum((a - e) * np.log(a / e))

def classer(x):
    if x < 0.10: return "stable"
    if x < 0.25: return "à surveiller"
    return "dérive"

variables = ["age", "anciennete", "encours", "ecart_taux"]
psi_vars = {v: psi(train_t[v], test_t[v]) for v in variables}
statut = {v: classer(x) for v, x in psi_vars.items()}
print(pd.DataFrame({"PSI": psi_vars, "statut": statut}).round(3))

              PSI  statut
age         0.022  stable
anciennete  0.008  stable
encours     0.007  stable
ecart_taux  6.965  dérive


In [12]:
verifier("Q4 · PSI(train, train) ≈ 0", lambda: psi(train_t["age"], train_t["age"]) < 1e-6)
verifier("Q4 · PSI de ecart_taux", lambda: np.isclose(psi_vars["ecart_taux"], 6.964637, atol=0.02))
verifier("Q4 · age et encours stables", lambda: statut["age"] == "stable" and statut["encours"] == "stable")
verifier("Q4 · dérive détectée sur ecart_taux", lambda: statut["ecart_taux"] == "dérive", "le niveau de taux a changé en 2023")

✅ Q4 · PSI(train, train) ≈ 0
✅ Q4 · PSI de ecart_taux
✅ Q4 · age et encours stables
✅ Q4 · dérive détectée sur ecart_taux


**Q4c · QCM.** La dérive de `ecart_taux` est détectée : quelle action ? **A.** ne rien faire (le modèle reste bon) · **B.** déclencher une revue : réentraînement, recalibration, communication à la validation · **C.** supprimer la variable. (`action` = `"A"`, `"B"` ou `"C"`)

In [13]:
action = "B"

In [14]:
verifier("Q4c · action de monitoring", lambda: action == "B", "le réentraînement est une étape planifiée, pas un accident")

✅ Q4c · action de monitoring


**Correction** — Le PSI de `ecart_taux` dépasse largement 0,25 : le contexte de taux a changé (c'est exactement ce qui explique la sous-estimation de la partie C). Les autres variables sont stables : la dérive est **localisée**, ce qui oriente le diagnostic. Le monitoring ne corrige rien tout seul : il **déclenche** une décision de gouvernance.

In [15]:
bilan()


🏁 Score : 19/19 vérifications réussies
   Bravo, exercice terminé !
